# Human-in-the-Loop: 사전 실행 게이트, 위험 등급 분류 및 감사 로그 기록

이 수업의 README는 에이전트가 이미 응답을 생성한 후 사용자에게 `APPROVE` 또는 `REJECT`를 묻는 짧은 코드 조각으로 Human-in-the-Loop를 소개합니다. 이 패턴은 좋은 출발점이지만, 실제 생산 환경의 HITL 구현에서는 일반적으로 세 가지 추가 요소가 필요합니다:

1. 위험한 단계를 에이전트가 실행하기 <strong>전에</strong> 동작하는 <strong>사전 실행 게이트</strong>로, 비용, 비가역성 및 지연 시간을 관리합니다.
2. <strong>위험 등급 분류</strong>로 저위험 작업은 자동 실행되고, 중간 위험 작업은 일괄 승인되며, 고위험 작업만 사람이 중단합니다.
3. <strong>감사 로그와 수정 루프</strong>로, 모든 게이트 결정이 JSONL 형식으로 기록되며 거부 시 단순히 `Revising...`을 출력하는 대신 구조화된 이유와 함께 에이전트를 다시 호출합니다.

이 노트북은 `06-system-message-framework.ipynb`와 동일한 원시 기능을 기반으로 각 요소를 구축합니다. `DEMO_MODE = True`에서는 인터랙티브 입력 없이도 처음부터 끝까지 실행되며, `DEMO_MODE = False`일 때는 실제 `input()` 프롬프트를 사용합니다. 참고: DEMO_MODE에서는 세 번째 목표의 재시도가 스크립트화되어 루프 동작을 끝까지 확인할 수 있습니다. 실제 수정 기반 재분류에는 `DEMO_MODE = False` 및 운영자가 필요합니다.

**범위 외 (다른 수업에서 다룸):** 인증 및 접근 제어 (Lesson 06 README 위협 2), 도구 호출 미들웨어 (Lesson 14 MAF 심층 분석), 다중 에이전트 토론 패턴.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## 패턴 1: 사전 실행 게이트

README의 HITL 코드 조각은 에이전트를 먼저 호출한 다음, 사용자가 출력을 승인하도록 요청합니다. 이것은 **사후 실행** 흐름입니다. 에이전트가 이미 실행되었으므로 LLM 호출 비용이 이미 지불되었고, 모든 부작용(발송된 이메일, 기록된 데이터베이스 행, 게시된 댓글)이 이미 발생한 상태입니다.

**사전 실행** 흐름은 에이전트가 위험한 단계를 실행하기 전에 게이트를 삽입합니다. 에이전트가 행동을 제안하면, 게이트가 실행 여부를 결정하고 승인된 경우에만 부작용이 발생합니다.

| 측면 | 사후 실행 승인 (README 코드 조각) | 사전 실행 게이트 (이 노트북) |
|---|---|---|
| 승인은 언제 실행되나요? | 에이전트가 출력을 생성한 후 | 어떤 부작용도 실행되기 전에 |
| 거절 시 LLM 비용 | 이미 지불됨 | 제안에 대해서만 지불, 행동에 대해서는 지불하지 않음 |
| 돌이킬 수 없는 부작용 | 가능 (이미 행동이 발생함) | 방지됨 |
| 감사 명확성 | 승인은 출력문임 | 승인은 타임스탬프, 행동, 이유가 포함된 JSON 기록임 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## 패턴 2: 위험 등급 구분

모든 작업에 인간의 승인이 필요한 것은 아닙니다. 공개 API에 대한 읽기 전용 조회는 고객 이메일 발송과는 다른 위험을 가집니다. 둘을 똑같이 처리하면 운영자의 주의가 낭비되고 에이전트가 느려집니다.

간단한 3단계 모델:

| 등급 | 예시 | 승인 흐름 |
|---|---|---|
| `low` (읽기 전용) | 지식 베이스 검색, 항공편 옵션 조회, 공개 웹 페이지 가져오기 | 자동 실행, 감사용으로 기록 |
| `medium` (저비용 변동) | 결과 캐시, 카운터 증가, 알림 예약 | 자동 실행, 일일 검토로 일괄 처리 |
| `high` (외부 노출 또는 되돌릴 수 없는 작업) | 이메일 발송, 카드 결제, 공개 채널 게시 | 인간 승인 대기 |

이것이 하나의 등급 구분 방법입니다. 실제 시스템에서는 더 세분화된 등급을 사용하는 경우가 많습니다(예: AWS IAM 권한 수준, 역할 기반 접근 등급). 아래 3단계 버전은 읽기 전용 및 부수 효과 작업을 혼합한 에이전트에 가장 적합한 최소한의 버전입니다.

아래 분류기는 데모가 결정론적이고 저렴하게 유지되도록 키워드 휴리스틱을 사용합니다. 실제 시스템에서는 학습된 분류기나 정책 엔진으로 대체할 수 있습니다.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 패턴 3: 감사 로그 및 수정 루프

`print("Response approved.")`는 감사 로그가 아닙니다. 신뢰를 위해, 모든 게이트 결정은 나중에 조회하거나 재생하거나 사고 검토에 첨부할 수 있는 구조화된 이벤트로 기록되어야 합니다.

두 가지 요소:

1. **추가 전용 JSONL.** 결정마다 한 줄씩, 타임스탬프, 액션, 등급, 결정, 이유 포함. grep하기 쉽고 나중에 실제 로그 저장소로 전송하기 쉽습니다.
2. **거부 시 수정 루프.** 게이트가 `deny`를 반환하면, 에이전트는 거부 이유를 문맥에 포함하여 스스로 다시 프롬프트를 보내 다음 제안이 문제를 피할 수 있게 합니다.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 추가 자료

다른 여러 공개 프로젝트들도 이러한 HITL 패턴의 변형을 구현합니다. 다양한 접근 방식을 비교하여 본인의 스택에 맞는 것을 찾아보세요:

- **LangChain** 인간 참여 도구 래퍼 ([문서](https://python.langchain.com/docs/integrations/tools/human_tools)): 실행을 중단하고 인간의 입력을 대기하는 드롭인 도구 래퍼입니다.
- **AutoGen** `UserProxyAgent` ([v0.2 문서](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+에서 구조가 변경됨): 다중 에이전트 대화에서 인간을 나타내는 역할로 에이전트를 사용합니다.
- **Semantic Kernel** 함수 필터 ([문서](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): 모든 함수 호출 주위에 실행되는 미들웨어로, 게이팅 로직에 적합합니다.

각 프로젝트는 세 가지 하위 패턴을 다르게 처리합니다: LangChain은 도구로 래핑하고, AutoGen은 에이전트 역할을 사용하며, Semantic Kernel은 미들웨어 필터를 사용합니다. 자신의 에이전트를 설계하기 전에 한두 가지 구현을 처음부터 끝까지 읽어보세요.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
